In [14]:
# ============================================
# 🧰 Install dependencies
# ============================================
!pip install pybloom_live tqdm pandas msgspec polars pyarrow networkit numpy scipy nltk --quiet

# Download NLTK tokenizer
import nltk
nltk.download('punkt', quiet=True)

# ============================================
# 📥 Clone RedPajama-Data source and add sys.path
# ============================================
!git clone https://github.com/togethercomputer/RedPajama-Data.git -q

import sys, os
sys.path.append("/content/RedPajama-Data/app/src")

from utilities.text.normalization import normalize
from utilities.text.ngrams import form_ngrams
from dedupe.utils import optimal_param
from dedupe.minhash import MinHash as RPMinHash

# 👉 Introducing RedPajama's Document and quality_signals registration functions.
from core.document import Document
from core.quality_signals.content import register_content_callables
from core.quality_signals.lines import register_lines_callables
from core.quality_signals.natural_language import register_natural_language_callables
from core.quality_signals.repetitions import register_repetitions_callables
from core.quality_signals.classifiers import register_classifier_callables
from core.quality_signals.importance_weights import register_importance_weights_callables

# ============================================
# 📦 Paths and configuration
# ============================================
from pathlib import Path

DATA_PATH = "/content/final_uniform_replace.jsonl"
MYTEXT_PATH = "/content/myText.txt"

EXACT_OUT_DIR = Path("/content/rp_exact_dedup_output")
EXACT_OUT_DIR.mkdir(parents=True, exist_ok=True)
LSH_OUT_DIR = Path("/content/rp_lsh_output")
LSH_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ✅ Use the artifacts directory that comes with the RedPajama repository
ARTIFACTS_DIR = Path("/content/RedPajama-Data/artifacts")
BAD_WORDS_DIR = ARTIFACTS_DIR / "bad_words"
BAD_URLS_DIR = ARTIFACTS_DIR / "bad_urls"
CLASSIFIERS_DIR = ARTIFACTS_DIR / "classifiers"
DSIR_DIR = ARTIFACTS_DIR / "dsir"

# Here an empty dict is used to simulate the case where there is no dsir / classifier file.
# Calls are made in the same way as the official worker, except that the model is not actually loaded.
EMPTY_DSIr_FILES = {}
EMPTY_CLASSIFIER_FILES = {}

# ============================================
# 💧 Load invisible characters from myText.txt
# ============================================
with open(MYTEXT_PATH, "r", encoding="utf-8") as f:
    chars = f.read()
INVISIBLE_CHARS = [c for c in chars if not c.isprintable() and c != "\n"]
INVISIBLE_CODES = [f"U+{ord(c):04X}" for c in INVISIBLE_CHARS]
print(f"💧 Loaded {len(INVISIBLE_CHARS)} invisible watermark characters:")
print(", ".join(INVISIBLE_CODES[:10]) + (" ..." if len(INVISIBLE_CODES) > 10 else ""))

def count_invisible(text: str, idx: int) -> int:
    """
    Count only the specific invisible character for this line index.
    Note: It is assumed that only one character, INVISIBLE_CHARS[idx], is used as the watermark for the idx sample.
    """
    if idx >= len(INVISIBLE_CHARS):
        return 0
    ch = INVISIBLE_CHARS[idx]
    return text.count(ch)

# ============================================
# 🧼 Stage 0: RedPajama-style quality signals / cleaning
# ============================================

def init_rp_quality_callables(
    lang: str = "en",
    ldnoobw_dir: Path = BAD_WORDS_DIR,
    ut1_dir: Path = BAD_URLS_DIR,
):
    """
    Modeled after core.worker.Worker.__init_quality_signals,
    but do try/except for missing files and use as much as you can.
    """
    callables = []

    # 1) content signal (depends on bad_words / bad_urls)
    try:
        callables += register_content_callables(
            language=lang,
            bad_urls_dir=ut1_dir,
            bad_words_dir=ldnoobw_dir,
        )
    except FileNotFoundError as e:
        print(f"[RP] Skipping content signals: {e}")

    # 2) Repeatable signals
    try:
        callables += register_repetitions_callables()
    except Exception as e:
        print(f"[RP] Skipping repetitions signals: {e}")

    # 3) Natural Language / Unnatural Language Signals
    try:
        callables += register_natural_language_callables()
    except Exception as e:
        print(f"[RP] Skipping natlang signals: {e}")

    # 4) Line Level Signal
    try:
        callables += register_lines_callables()
    except Exception as e:
        print(f"[RP] Skipping line-level signals: {e}")

    # 5) fastText classifier signals (if there is no .bin it will give an error, just skip it)
    try:
        wikiref_model = next((p for p in (CLASSIFIERS_DIR / lang).glob("wikiref*.bin")), None) \
            if (CLASSIFIERS_DIR / lang).exists() else None
        palm_model = next((p for p in (CLASSIFIERS_DIR / lang).glob("palm*.bin")), None) \
            if (CLASSIFIERS_DIR / lang).exists() else None
        wikipedia_model = next((p for p in (CLASSIFIERS_DIR / lang).glob("wikipedia*.bin")), None) \
            if (CLASSIFIERS_DIR / lang).exists() else None

        callables += register_classifier_callables(
            wikiref_model=str(wikiref_model) if wikiref_model else None,
            palm_model=str(palm_model) if palm_model else None,
            wikipedia_model=str(wikipedia_model) if wikipedia_model else None,
        )
    except FileNotFoundError as e:
        print(f"[RP] Skipping classifier signals: {e}")
    except Exception as e:
        print(f"[RP] Skipping classifier signals (other error): {e}")

    # 6) DSIR importance weighting (skip if no .npy file)
    try:
        def collect_dsir(domain: str):
            dom_dir = DSIR_DIR / lang
            if not dom_dir.exists():
                return None
            counts = list(dom_dir.glob(f"{domain}.counts.npy"))
            lambdas = list(dom_dir.glob(f"{domain}.lambda.npy"))
            if not counts or not lambdas:
                return None
            return [str(counts[0]), str(lambdas[0])]

        callables += register_importance_weights_callables(
            source_fps=collect_dsir("ccnet"),
            wiki_fps=collect_dsir("wikipedia"),
            openwebtext_fps=collect_dsir("openwebtext"),
            books_fps=collect_dsir("books"),
            language=lang,
        )
    except FileNotFoundError as e:
        print(f"[RP] Skipping DSIR signals: {e}")
    except Exception as e:
        print(f"[RP] Skipping DSIR signals (other error): {e}")

    return callables

RP_QUALITY_CALLABLES = init_rp_quality_callables(lang="en")

def compute_rp_quality_signals(text: str, language: str = "en") -> dict:
    """
    Using RedPajama's official Document + quality_signals callables,
    computes all RP signals for a given text and returns a dict.
    """
    document = Document(
        text,
        domain="",
        precompute_ngrams=True,
        precompute_hash_features=True,
        dsir_buckets=10_000,       # Consistent with Worker
    )

    signals = {}
    for func in RP_QUALITY_CALLABLES:
        try:
            signals[func.field_name] = func(document)
        except Exception as e:
            signals[func.field_name] = f"ERROR: {e.__class__.__name__}"

    return signals

# ============================================
# 🔹 Stage 1️⃣: Exact Deduplication (Bloom Filter)
# ============================================
import hashlib, json, msgspec
from pybloom_live import BloomFilter
from tqdm import tqdm

capacity = 1_000_000
error_rate = 1e-4
unique_fp = EXACT_OUT_DIR / "unique.jsonl"
duplicates_fp = EXACT_OUT_DIR / "duplicates.jsonl"
bloom_fp = EXACT_OUT_DIR / "bloomfilter.pkl"  # Optional: If you want to persist it, you can store it in a pickle.

bloom = BloomFilter(capacity=capacity, error_rate=error_rate)
print(f"🧮 Bloom Filter initialized. capacity={capacity}, error_rate={error_rate}")

def compute_digest(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

num_docs = 0
num_dupes = 0

with open(DATA_PATH, "r", encoding="utf-8") as fin, \
     open(unique_fp, "w", encoding="utf-8") as fout_unique, \
     open(duplicates_fp, "w", encoding="utf-8") as fout_dupes:
    for line in tqdm(fin, desc="Processing documents"):
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue
        text = record.get("watermarked", "").strip()
        if not text:
            continue

        # ===== 🧼 Call RedPajama official cleaning / quality pipeline =====
        rp_signals = compute_rp_quality_signals(text, language="en")
        record["rp_quality_signals"] = rp_signals
        # If you want to actually filter based on these signals in the future, you can write rules here, for example:
        # if rp_signals["rps_doc_is_natural_language"][0][2] < 0.5: continue
        # Currently this code just calculates and saves the signal, it does not do hard filter.

        digest = compute_digest(text)
        num_docs += 1
        if digest in bloom:
            num_dupes += 1
            fout_dupes.write(json.dumps({"digest": digest, **record}, ensure_ascii=False) + "\n")
        else:
            bloom.add(digest)
            fout_unique.write(json.dumps({"digest": digest, **record}, ensure_ascii=False) + "\n")

print("✅ Exact Dedup complete!")
print(f"Total: {num_docs:,}, Duplicates: {num_dupes:,}, Unique: {num_docs - num_dupes:,}")
print(f"Results saved in: {EXACT_OUT_DIR}")

# (Optional) Persist BloomFilter to file
import pickle
with open(bloom_fp, "wb") as f:
    pickle.dump(bloom, f)

# ============================================
# 💧 Watermark retention check (exact dedup, 1-to-1 character mapping)
# ============================================
total_wm = 0
unique_wm_retained = 0
dupe_wm_removed = 0

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        if rec.get("is_watermarked", False):
            total_wm += 1

with open(unique_fp, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        if rec.get("is_watermarked", False):
            text = rec.get("watermarked", "")
            if count_invisible(text, i) > 0:
                unique_wm_retained += 1

with open(duplicates_fp, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        if rec.get("is_watermarked", False):
            text = rec.get("watermarked", "")
            if count_invisible(text, i) > 0:
                dupe_wm_removed += 1

if total_wm > 0:
    retention_rate = unique_wm_retained / total_wm * 100
    lost_rate = dupe_wm_removed / total_wm * 100
else:
    retention_rate = lost_rate = 0

print("\n=================== 💧 1-to-1 Watermark Retention after Exact Dedup ===================")
print(f"Total watermarked docs: {total_wm}")
print(f"Retained (unique) docs w/ watermark: {unique_wm_retained}")
print(f"Removed (duplicate) docs w/ watermark: {dupe_wm_removed}")
print(f"✅ Retention rate: {retention_rate:.2f}% | ⚠️ Lost rate: {lost_rate:.2f}%")
print("======================================================================================")

# ============================================
# 🔸 Stage 2️⃣: RedPajama-style Fuzzy Dedup (MinHash + LSH)
# Using unique.jsonl as input
# ============================================
print("\n🚀 Entering LSH fuzzy deduplication stage...")
NEW_DATA_PATH = str(unique_fp)   # Input is the unique file from exact dedup

# ============================================
# 🧩 Paths and parameters (RedPajama-style)
# ============================================
DATA_PATH = NEW_DATA_PATH
OUT_DIR = LSH_OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

SIMILARITIES = [0.7]
NUM_PERM = 128         # number of permutations
NGRAM_SIZE = 3         # token 3-gram
SEED = 42

# ============================================
# 📦 Build MinHash parquet file (RedPajama-compatible format)
# Each row: id, id_int, shard_id, signature_sim0.7 (list[bytes])
# ============================================
import polars as pl

def sha1_64_int(doc_id: str) -> int:
    """Convert document ID to 64-bit integer (same as RedPajama worker.py)."""
    byteorder = sys.byteorder  # usually 'little'
    return int.from_bytes(
        hashlib.sha1(doc_id.encode("utf-8")).digest()[:8],
        byteorder=byteorder,
        signed=False,
    )

def build_minhash_parquet(
    data_path: str,
    out_parquet: Path,
    similarities: list,
    num_perm: int,
    ngram_size: int,
    seed: int,
    shard_id: str = "local/0000",
):
    """Generate MinHash signatures for all docs and save to a Parquet file using RedPajama's MinHash."""
    mh = RPMinHash(
        similarity_thresholds=similarities,
        ngram_size=ngram_size,
        num_permutations=num_perm,
        seed=seed,
    )

    ids, id_ints = [], []
    sig_cols = {f"signature_sim{s}": [] for s in similarities}

    # Read JSONL line by line
    with open(data_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            text = rec.get("watermarked", "")
            if not text:
                continue

            # RedPajama-compatible normalize + tokenize
            norm = normalize(text)
            tokens = tuple(norm.split())  # equivalent to Document.normalized_words for words

            # Compute signatures via RedPajama MinHash
            sigs = mh.compute_banded_signatures(tokens)

            doc_id = f"{shard_id}/{i}"        # matches worker-style ID
            id_int = sha1_64_int(doc_id)

            ids.append(doc_id)
            id_ints.append(id_int)
            for s in similarities:
                key = f"signature_sim{s}"
                sig_cols[key].append(sigs[key])

    # Assemble Polars DataFrame
    df = pl.DataFrame(
        {
            "id": ids,
            "id_int": id_ints,
            "shard_id": [shard_id] * len(ids),
            **sig_cols,
        }
    )

    df.write_parquet(out_parquet)
    return out_parquet

minhash_parquet = OUT_DIR / "minhash.parquet"
_ = build_minhash_parquet(
    data_path=DATA_PATH,
    out_parquet=minhash_parquet,
    similarities=SIMILARITIES,
    num_perm=NUM_PERM,
    ngram_size=NGRAM_SIZE,
    seed=SEED,
    shard_id="local/0000",
)
print(f"✅ MinHash parquet written: {minhash_parquet}")

# ============================================
# 🧭 LSH-based fuzzy deduplication (RedPajama run_lsh.py logic)
# Group by band → connect same-bucket pairs → compute connected components
# ============================================
import pyarrow.dataset as ds
import networkit.components as nk_components
import networkit.graph as nk_graph
import numpy as np

def run_rp_lsh(
    minhash_parquets: list,
    out_dir: Path,
    similarity: float,
    num_perm: int,
    max_docs: int = -1,
):
    """Run RedPajama-style LSH fuzzy deduplication."""
    sig_key = f"signature_sim{similarity}"

    # Load Parquet dataset
    try:
        dset = ds.dataset(source=minhash_parquets, format="parquet")
    except TypeError:
        dset = ds.dataset(minhash_parquets, format="parquet")

    # Scan and select only necessary columns
    query = (
        pl.scan_pyarrow_dataset(dset)
        .select(pl.col(["id_int", sig_key]))
        .filter(~pl.col(sig_key).is_null())
    )

    # Compute (bands, rows) for this similarity using RedPajama's optimal_param
    b, r = optimal_param(similarity, num_perm)
    num_bands = b

    # Expand by band and bucketize
    query = (
        query
        .with_columns(pl.lit(list(range(num_bands))).alias("band"))
        .explode(sig_key, "band")
        .group_by(sig_key, "band")
        .agg(pl.col("id_int"))
    )

    # Keep only buckets with more than one doc
    try:
        query = query.filter(pl.col("id_int").list.lengths() > 1)
    except AttributeError:
        query = query.filter(pl.col("id_int").list.len() > 1)

    # Create edges between docs sharing same bucket
    query = (
        query
        .select(
            pl.col("id_int"),
            pl.col("id_int").list.min().alias("min_node"),
        )
        .explode("id_int")
        .filter(pl.col("id_int") != pl.col("min_node"))
        .select(pl.concat_list(["id_int", "min_node"]).alias("edges"))
        .unique("edges")
    )

    # Collect edges as list of [id_int, min_node]
    edges_df = query.collect()
    edges = edges_df["edges"].to_list()  # list of [n1, n2]
    print(f"🧱 Edges: {len(edges)} total")

    # Build undirected graph
    graph = nk_graph.Graph()
    node_map = {}
    for n1, n2 in edges:
        if n1 not in node_map:
            node_map[n1] = graph.addNode()
        if n2 not in node_map:
            node_map[n2] = graph.addNode()
        graph.addEdge(node_map[n1], node_map[n2])

    rev_map = {v: k for k, v in node_map.items()}

    # Compute connected components
    cc = nk_components.ConnectedComponents(G=graph)
    cc.run()
    comps = cc.getComponents()
    print(f"🔗 Connected components (clusters): {len(comps)}")

    # Convert to doc → cluster_id mapping
    def process_comp(comp):
        if not comp:
            return np.empty((0, 2), dtype=np.uint64)  # skip empty clusters
        nodes = np.array(list(map(rev_map.get, comp))).reshape(-1, 1)
        cid = np.repeat(min(map(rev_map.get, comp)), len(nodes)).reshape(-1, 1)
        return np.hstack((nodes, cid))

    if len(comps) > 0:
        data = np.vstack(tuple(map(process_comp, comps)))
        clusters_df = pl.DataFrame(data, schema=["id_int", "cluster_id"])
    else:
        clusters_df = pl.DataFrame({"id_int": [], "cluster_id": []})

    out_path = out_dir / f"clusters_sim{similarity}.parquet"
    clusters_df.write_parquet(out_path)
    print(f"✅ Clusters written: {out_path}")
    return out_path

# Run LSH fuzzy deduplication
clusters_parquet = run_rp_lsh(
    minhash_parquets=[minhash_parquet],
    out_dir=OUT_DIR,
    similarity=SIMILARITIES[0],
    num_perm=NUM_PERM,
    max_docs=-1,
)

# ============================================
# 💧 LSH watermark detect
# ============================================
clusters_parquet = str(LSH_OUT_DIR / "clusters_sim0.7.parquet")
clusters = pl.read_parquet(clusters_parquet)

def doc_id_of(i):
    return f"local/0000/{i}"

def id_int_of(i):
    return int.from_bytes(
        hashlib.sha1(doc_id_of(i).encode("utf-8")).digest()[:8],
        byteorder=sys.byteorder,
        signed=False,
    )

wm_id_ints = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        if rec.get("is_watermarked", False):
            text = rec.get("watermarked", "")
            if count_invisible(text, i) > 0:
                wm_id_ints.append(id_int_of(i))

wm_total = len(wm_id_ints)
print(f"\n📘 Total watermarked documents: {wm_total}")

# Identify duplicates in clusters
if clusters.height > 0:
    rep = clusters.group_by("cluster_id").agg(
        pl.col("id_int").min().alias("rep_id")
    )
    clusters = clusters.join(rep, on="cluster_id", how="left")
    clusters = clusters.with_columns(
        (pl.col("id_int") != pl.col("rep_id")).alias("is_duplicate_member")
    )
else:
    clusters = clusters.with_columns(
        pl.Series(name="rep_id", values=[]),
        pl.Series(name="is_duplicate_member", values=[]),
    )

if clusters.is_empty():
    dupe_id_ints = set()
else:
    dupe_id_ints = set(
        clusters.filter(pl.col("is_duplicate_member"))
        .get_column("id_int")
        .to_list()
    )

wm_dupes = sum(1 for _id in wm_id_ints if _id in dupe_id_ints)
wm_retained = wm_total - wm_dupes
ret_rate = (wm_retained / wm_total * 100) if wm_total > 0 else 0.0

print("====== 💧 1-to-1 Watermark Retention after LSH Dedup ======")
print(f"Removed watermarked docs: {wm_dupes}")
print(f"Retained watermarked docs: {wm_retained}")
print(f"✅ Retention rate: {ret_rate:.2f}%")
print("===========================================================")
